In [1]:
import surprise
import pandas as pd
import numpy as np
import os
os.chdir("E:/Machine_Learning/Cases_Rec_Sys/ml-100k/")

In [2]:
ratings = pd.read_csv('u.data',sep="\t",names=['uid','iid','rating','ts'])
ratings.drop('ts',axis=1,inplace=True)
ratings.head()

,uid,iid,rating
0,196,242,3
1,186,302,3
2,22,377,1
3,244,51,2
4,166,346,1


In [3]:
lowest_rating=ratings['rating'].min()
highest_rating=ratings['rating'].max()
lowest_rating,highest_rating

(np.int64(1), np.int64(5))

In [4]:
reader = surprise.Reader(rating_scale = (lowest_rating,highest_rating))
data = surprise.Dataset.load_from_df(ratings,reader)
type(data)

surprise.dataset.DatasetAutoFolds

In [5]:
similarity_options = {'name' :'cosine','user_based': True}
algo = surprise.KNNBasic(sim_options = similarity_options)  #Default k = 40
output = algo.fit(data.build_full_trainset())
#Calculates the expected ratings for all the users

Computing the cosine similarity matrix...
Done computing similarity matrix.


In [6]:
from surprise.model_selection import train_test_split

In [7]:
trainset , testset = train_test_split(data,test_size = 0.25,random_state=26)
similarity_options = {'name' :'cosine','user_based': True}
algo = surprise.KNNBasic(sim_options = similarity_options,k=30) 
algo.fit(trainset)
predictions = algo.test(testset)
surprise.accuracy.rmse(predictions)

Computing the cosine similarity matrix...
Done computing similarity matrix.
RMSE: 1.0294


np.float64(1.02943178809445)

## GridSearchCV

In [8]:
from surprise.model_selection import GridSearchCV
from surprise.model_selection import KFold

In [9]:
param_grid = { 'k': [25,40,50,65,75,85],'user_based': [False,True]}
kFold = KFold(n_splits= 5,random_state=26,shuffle=True)
gs = GridSearchCV(surprise.KNNBasic,param_grid,measures=['rmse','mae'],cv=kFold)
gs.fit(data)

Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computi

In [10]:
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])

0.9761915086082429
{'k': 25, 'user_based': False}


In [11]:
algo = gs.best_estimator['rmse']
algo.fit(data.build_full_trainset())

Computing the msd similarity matrix...
Done computing similarity matrix.


 ## Tota number of items

In [12]:
iids = ratings['iid'].unique()
print(len(iids))

1682


### Consider any user

In [13]:
a = [2,4,5,7,8,9]
b = [4,7,9]
np.setdiff1d(a,b)

array([2, 5, 8])

In [14]:
user = 50
u_iid = ratings[ratings['uid'] ==user]['iid'].unique()
print("list of items rated by user : \n",u_iid)
print("No. of items rated by user {0} : {1} ".format(user,len(u_iid)))
iids_to_predict = np.setdiff1d(iids,u_iid)
print("items not rated by the user or those items which the expected ratings are to be predicted : \n")
print(iids_to_predict)

list of items rated by user : 
 [ 246  823  253  475 1084  286    9  125  123  325  508  288  319  324
  276 1008 1010  268  544   15  327  124  547  100]
No. of items rated by user 50 : 24 
items not rated by the user or those items which the expected ratings are to be predicted : 

[   1    2    3 ... 1680 1681 1682]


In [15]:
testset = [[user,iid,0.] for iid in iids_to_predict]
predictions = algo.test(testset)
exp_ratings = [(predictions[i].iid,predictions[i].est) for i in range(0,len(predictions))]
df_exp_ratings = pd.DataFrame(exp_ratings,columns=['iid','est_rating'])
df_exp_ratings.sort_values(by='est_rating',ascending=False).head(30)

,iid,est_rating
1164,1189,5.000000
1176,1201,5.000000
1097,1122,5.000000
793,814,5.000000
1475,1500,5.000000
1574,1599,5.000000
1511,1536,5.000000
1442,1467,5.000000
1628,1653,5.000000
1268,1293,5.000000


## Movies list dataset

In [16]:
movies = pd.read_csv('movies_list.csv',encoding = 'latin-1')
movies.columns

Index(['movie id ', ' movie title ', ' release date ', ' video release date ',
       'IMDb URL ', ' unknown ', ' Action ', ' Adventure ', ' Animation ',
       'Children's ', ' Comedy ', ' Crime ', ' Documentary ', ' Drama ',
       ' Fantasy ', 'Film-Noir ', ' Horror ', ' Musical ', ' Mystery ',
       ' Romance ', ' Sci-Fi ', 'Thriller ', ' War ', ' Western'],
      dtype='object')

In [23]:
df_exp_ratings = df_exp_ratings.merge(movies[['movie id ', ' movie title ', ' release date ']],how = 'left',left_on='iid',right_on='movie id ')
df_exp_ratings.sort_values(by = 'est_rating',ascending = False).head(10)

,iid,est_rating,movie id _x,movie title _x,release date _x,movie id _y,movie title _y,release date _y
1164,1189,5.0,1189,Prefontaine (1997),24-Jan-97,1189,Prefontaine (1997),24-Jan-97
1176,1201,5.0,1201,Marlene Dietrich: Shadow and Light (1996),02-Apr-96,1201,Marlene Dietrich: Shadow and Light (1996),02-Apr-96
1097,1122,5.0,1122,They Made Me a Criminal (1939),01-Jan-39,1122,They Made Me a Criminal (1939),01-Jan-39
793,814,5.0,814,"Great Day in Harlem, A (1994)",01-Jan-94,814,"Great Day in Harlem, A (1994)",01-Jan-94
1475,1500,5.0,1500,Santa with Muscles (1996),08-Nov-96,1500,Santa with Muscles (1996),08-Nov-96
1574,1599,5.0,1599,Someone Else's America (1995),10-May-96,1599,Someone Else's America (1995),10-May-96
1511,1536,5.0,1536,Aiqing wansui (1994),22-Jul-96,1536,Aiqing wansui (1994),22-Jul-96
1442,1467,5.0,1467,"Saint of Fort Washington, The (1993)",01-Jan-93,1467,"Saint of Fort Washington, The (1993)",01-Jan-93
1628,1653,5.0,1653,Entertaining Angels: The Dorothy Day Story (1996),27-Sep-96,1653,Entertaining Angels: The Dorothy Day Story (1996),27-Sep-96
1268,1293,5.0,1293,Star Kid (1997),16-Jan-98,1293,Star Kid (1997),16-Jan-98
